# Operational Outputs

Operational outputs translate the daily H3 prediction cube into products that are easier to review, communicate, and compare through time. The purpose is not to create a new model. The purpose is to repackage the same species-use and risk surfaces into weekly, fisheries-grid, gear-aware, and animation-ready views.

These products are especially important for this project because they connect the modeling workflow to practical screening questions: which weeks matter, which areas recur across years, and how do species-specific risk surfaces overlap with fishing activity?

## Product Families

The operational layer includes:

- weekly latent-risk climatology across 2014-2023;
- a 2022 weekly sequence for animation-oriented review;
- fisheries-grid aggregation of H3 weekly latent risk;
- vessel-activity overlays for operational interpretation;
- gear-aware realized-risk examples;
- videos generated from weekly sequence frames.

These are aggregations and visual translations of the prediction cube, not separate models.

## Weekly Latent-Risk Products

The weekly products are built from the active hybrid prediction partitions. The builder writes three parquet products:

```text
data/
└── modeling/
    └── weekly_operator/
        └── hybrid_presence_gate_extra_trees_som_hierarchical_k30_5fold_blockcv_bayesian_gmm_k30/
            └── joint/
                ├── latent_risk_iso_week_by_year_2014-2023.parquet
                ├── latent_risk_iso_week_climatology_2014-2023.parquet
                └── latent_risk_iso_week_sequence_2022.parquet
```

The weekly latent-risk score is computed from the weekly mean species-use surface plus a standardized minimum fishing-effort unit:

$$
latent\_risk\_log\_pred = species\_use\_log\_pred + \log(1 + minimum\_effort\_unit)
$$

The product also carries a display field that is set to missing where the weekly species-use mean is below the shared display threshold. That prevents very low species-use cells from dominating operational maps.

In code, the builder first averages daily species use by `h3`/`species`/ISO year/ISO week, then adds the same minimum-effort baseline used for latent-risk maps:

```python
weekly_species_use = daily.groupby(["h3", "species", "iso_year", "iso_week"])[
    "species_use_log_pred"
].mean()
weekly_latent_risk = weekly_species_use + np.log1p(minimum_effort_unit)
```

These parquet products are built by `scripts/build/build_weekly_operator_latent_risk.py` from the daily hybrid prediction partitions. The default minimum effort unit and display threshold come from `src/riskscape/visualization/maps.py`.


## Weekly Climatology Maps

The weekly climatology averages the 2014-2023 weekly surfaces. This gives a recurring seasonal view rather than emphasizing one particular year.

<p>
  <img src="../plots/predictions/weekly_operator/hybrid_presence_gate_extra_trees_som_hierarchical_k30_5fold_blockcv_bayesian_gmm_k30_joint_latent_risk_iso_week_climatology_2014-2023_small_multiples.png" width="98%" />
</p>

The weekly climatology map is generated by `scripts/plots/plot_weekly_operator_latent_risk.py`, also reachable through `scripts/plots/plot_all_maps.py --group weekly`.


## Fisheries-Grid Summaries

The H3 weekly climatology can be aggregated to fisheries grid squares. This is useful because H3 is the modeling grid, while fisheries grids are easier to communicate as management or reporting units.

```text
data/
└── plot_exports/
    └── weekly_operator/
        └── hybrid_presence_gate_extra_trees_som_hierarchical_k30_5fold_blockcv_bayesian_gmm_k30/
            └── joint/
                └── latent_risk_iso_week_climatology_2014-2023_fisheries_grid.parquet
```

<p>
  <img src="../plots/predictions/weekly_operator/hybrid_presence_gate_extra_trees_som_hierarchical_k30_5fold_blockcv_bayesian_gmm_k30_joint_latent_risk_iso_week_climatology_2014-2023_fisheries_grid_example.png" width="98%" />
</p>

Fisheries-grid summaries are created by `scripts/plots/plot_weekly_operator_fisheries_grid_example.py`; the plot export is written under `data/plot_exports/weekly_operator/`.


## Vessel-Activity Overlays

The weekly latent-risk climatology can be shown together with cells that had fishing activity in a selected year. The overlay does not change latent risk; it helps readers compare recurring potential risk with observed fishing footprint.

<p>
  <img src="../plots/fishing_activity/hybrid_presence_gate_extra_trees_som_hierarchical_k30_5fold_blockcv_bayesian_gmm_k30_joint_latent_risk_iso_week_climatology_2014-2023_all_vessel_cells_2022.png" width="98%" />
</p>

The vessel-activity overlay is generated by `scripts/plots/plot_weekly_latent_risk_with_jigger_activity.py`. Despite the historical script name, the plotted overlay is driven by the gear/flag fishing-effort export and can show all vessels or a selected flag.


## Flag-Filtered Fishing Activity

The fishing-activity export used by the operational plots keeps both gear type and vessel flag. This allows the same exposure layer to be filtered in different ways without changing the species-use or latent-risk products.

```text
data/
└── plot_exports/
    └── fishing_activity/
        └── fishing_effort_by_gear_flag_2022.parquet
```

The key fields are `h3`, `date`, `gear_type`, `flag`, `fishing_hours`, and `vessel_count`. A flag filter can be used to mark only a selected fleet on top of the weekly latent-risk climatology, for example `--flag FLK`. If no flag is provided, the overlay marks all vessel activity.

The generated example below shows the weekly latent-risk climatology with 2022 FLK-flagged vessel-activity cells marked.

<p>
  <img src="../plots/fishing_activity/hybrid_presence_gate_extra_trees_som_hierarchical_k30_5fold_blockcv_bayesian_gmm_k30_joint_latent_risk_iso_week_climatology_2014-2023_flk_vessel_cells_2022.png" width="98%" />
</p>

The gear/flag fishing-effort table is exported by `scripts/tools/export_fishing_effort_by_gear_flag.py`. The FLK overlay is generated with `scripts/plots/plot_weekly_latent_risk_with_jigger_activity.py --flag FLK`.


## Gear-Aware Examples

Gear-aware examples join the 2022 hybrid species-use predictions with the separate `h3`/`date`/`gear_type`/`flag` fishing-activity export. They are examples of operational filtering, not new species-use models.

The retained examples are:

- BBAL with set longlines.
- SAFS with trawlers.

The weekly gear-aware map shows realized risk after filtering the exposure layer by these species/gear pairs.

<p>
  <img src="../plots/fishing_activity/gear_aware_weekly_realized_risk_fisheries_grid_example_non_zero_mean_2022.png" width="98%" />
</p>

The gear-aware examples are generated by `scripts/plots/plot_weekly_gear_aware_risk_examples.py`; the separate BBAL set-longline example is generated by `scripts/plots/plot_set_longline_bbal_risk_example.py`.


## Animation Products

The 2022 weekly sequence can be rendered as MP4 animations, one per species. Videos and intermediate frames are generated products and are not tracked in Git, but they are included in the plot archive and can be regenerated with the `videos` plot group.

```text
plots/
└── predictions/
    └── weekly_operator/
        ├── hybrid_presence_gate_extra_trees_som_hierarchical_k30_5fold_blockcv_bayesian_gmm_k30_joint_latent_risk_iso_week_sequence_2022_BBAL.mp4
        └── hybrid_presence_gate_extra_trees_som_hierarchical_k30_5fold_blockcv_bayesian_gmm_k30_joint_latent_risk_iso_week_sequence_2022_SAFS.mp4
```

For presentations, preview copies are available on Vimeo:

- [BBAL weekly latent-risk animation](https://vimeo.com/1194835782)
- [SAFS weekly latent-risk animation](https://vimeo.com/1194835781)

Animations are generated by `scripts/plots/plot_weekly_operator_latent_risk.py --make-animation`, or through `scripts/plots/plot_all_maps.py --group videos`.


In [3]:
from pathlib import Path
import pandas as pd
import yaml

config = yaml.safe_load(Path("../config.yaml").read_text())
data_dir = Path("..") / config["paths"]["data"]
plots_dir = Path("..") / config["paths"]["plots"]

model_name = "hybrid_presence_gate_extra_trees_som_hierarchical_k30_5fold_blockcv_bayesian_gmm_k30"
weekly_root = data_dir / "modeling" / "weekly_operator" / model_name / "joint"
weekly_export_root = data_dir / "plot_exports" / "weekly_operator" / model_name / "joint"

artifacts = {
    "weekly by-year product": weekly_root / "latent_risk_iso_week_by_year_2014-2023.parquet",
    "weekly climatology product": weekly_root / "latent_risk_iso_week_climatology_2014-2023.parquet",
    "weekly sequence product, 2022": weekly_root / "latent_risk_iso_week_sequence_2022.parquet",
    "fisheries-grid weekly export": weekly_export_root / "latent_risk_iso_week_climatology_2014-2023_fisheries_grid.parquet",
    "weekly climatology plot": plots_dir / "predictions" / "weekly_operator" / "hybrid_presence_gate_extra_trees_som_hierarchical_k30_5fold_blockcv_bayesian_gmm_k30_joint_latent_risk_iso_week_climatology_2014-2023_small_multiples.png",
    "fisheries-grid plot": plots_dir / "predictions" / "weekly_operator" / "hybrid_presence_gate_extra_trees_som_hierarchical_k30_5fold_blockcv_bayesian_gmm_k30_joint_latent_risk_iso_week_climatology_2014-2023_fisheries_grid_example.png",
    "gear-aware plot": plots_dir / "fishing_activity" / "gear_aware_weekly_realized_risk_fisheries_grid_example_non_zero_mean_2022.png",
    "flag-filtered fishing activity export": data_dir / "plot_exports" / "fishing_activity" / "fishing_effort_by_gear_flag_2022.parquet",
}

pd.DataFrame(
    [
        {
            "product": name,
            "relative_path": path.relative_to(Path("..")),
            "status": "present" if path.exists() else "missing",
        }
        for name, path in artifacts.items()
    ]
)

,product,relative_path,status
0,weekly by-year product,data/modeling/weekly_operator/hybrid_presence_...,present
1,weekly climatology product,data/modeling/weekly_operator/hybrid_presence_...,present
2,"weekly sequence product, 2022",data/modeling/weekly_operator/hybrid_presence_...,present
3,fisheries-grid weekly export,data/plot_exports/weekly_operator/hybrid_prese...,present
4,weekly climatology plot,plots/predictions/weekly_operator/hybrid_prese...,present
5,fisheries-grid plot,plots/predictions/weekly_operator/hybrid_prese...,present
6,gear-aware plot,plots/fishing_activity/gear_aware_weekly_reali...,present
7,flag-filtered fishing activity export,data/plot_exports/fishing_activity/fishing_eff...,present


## Regenerating Operational Outputs

The command list below is a compact rebuild checklist. Running the notebook cell only displays the commands; it does not execute them.

Use the commands in this order when rebuilding operational products from restored data:

- `build_weekly_operator_latent_risk.py` rebuilds the weekly parquet products from the daily hybrid prediction partitions.
- `plot_all_maps.py --group weekly` regenerates weekly latent-risk maps.
- `plot_all_maps.py --group gear` regenerates the gear-aware examples.
- `plot_all_maps.py --group videos` regenerates the weekly MP4 animations.
- `plot_weekly_latent_risk_with_jigger_activity.py --flag FLK` regenerates the FLK-flagged vessel-activity overlay.

The listed commands correspond to the build and plot scripts named in the sections above.


In [4]:
commands = [
    "python3 scripts/build/build_weekly_operator_latent_risk.py --overwrite",
    "python3 scripts/plots/plot_all_maps.py --group weekly",
    "python3 scripts/plots/plot_all_maps.py --group gear",
    "python3 scripts/plots/plot_all_maps.py --group videos",
    "python3 scripts/plots/plot_weekly_latent_risk_with_jigger_activity.py --flag FLK",
]

pd.DataFrame({"command": commands})

,command
0,python3 scripts/build/build_weekly_operator_la...
1,python3 scripts/plots/plot_all_maps.py --group...
2,python3 scripts/plots/plot_all_maps.py --group...
3,python3 scripts/plots/plot_all_maps.py --group...
4,python3 scripts/plots/plot_weekly_latent_risk_...
